<a href="https://colab.research.google.com/github/leticiaximena/presupuesto_cuenta_publica_2025/blob/main/An%C3%A1lisis_de_Variaciones_Cierre_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de Variaciones | Presupuesto Modificado VS Ejercido

In [ ]:
# ============================================
# PASO 1: Conectar Google Drive a Colab
# Esto permite acceder directamente a los archivos guardados
# en tu Drive (como el CSV exportado desde BigQuery en el
# análisis anterior sobre concentración del gasto) sin
# necesidad de subirlo manualmente cada vez.
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================
# PASO 2: Cargar el archivo CSV en un DataFrame
# Se importa la librería pandas (estándar para análisis de
# datos en Python) y se carga el archivo exportado de BigQuery.
# ============================================

import pandas as pd

# Ajusta la ruta según dónde se haya guardado el archivo en tu Drive
df = pd.read_csv('/content/drive/MyDrive/analisis_python/concentrado_registros.csv')

# Muestra las primeras 5 filas para confirmar que se cargó bien
df.head()

,ENTIDAD_FEDERATIVA,Estado,PROGRAMA_PRESUPUESTARIO,clave_obra,nom_obra,ORIGINAL,MODIFICADO,EJERCIDO
0,4,Campeche,K039,24092100004,Programa de Estudios y Proyectos de Caminos Ru...,0,24960.00,24960.00
1,6,Colima,K037,24092100003,Programa de Conservación de Infraestructura de...,0,7000.00,7000.00
2,6,Colima,K031,25096260004,"Construcción del Puente Vehicular La Presa, so...",0,579715.00,579713.14
3,6,Colima,K031,25096260001,Construcción del Paso Superior Vehicular Arco Sur,0,135000.00,135000.00
4,6,Colima,K031,25096260002,Construcción del Paso Superior Vehicular Arco ...,0,19165.21,19165.21


In [ ]:
# ============================================
# PASO 3: Validación inicial de los datos
# Antes de calcular variaciones, se revisa la estructura del
# DataFrame: cuántas filas y columnas tiene, qué tipo de dato
# tiene cada columna, y si existen valores nulos que deban
# tratarse antes del análisis.
# ============================================

# Dimensiones del DataFrame: (número de filas, número de columnas)
print("Dimensiones:", df.shape)

# Tipos de dato de cada columna
print("\nTipos de dato:")
print(df.dtypes)

# Conteo de valores nulos por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

Dimensiones: (301, 8)

Tipos de dato:
ENTIDAD_FEDERATIVA           int64
Estado                      object
PROGRAMA_PRESUPUESTARIO     object
clave_obra                   int64
nom_obra                    object
ORIGINAL                     int64
MODIFICADO                 float64
EJERCIDO                   float64
dtype: object

Valores nulos por columna:
ENTIDAD_FEDERATIVA          0
Estado                      0
PROGRAMA_PRESUPUESTARIO     0
clave_obra                  0
nom_obra                   12
ORIGINAL                    0
MODIFICADO                  0
EJERCIDO                    0
dtype: int64


In [ ]:
# ============================================
# PASO 4: Inspeccionar registros sin nombre de obra
# Se identifican las filas donde nom_obra es nulo, para
# entender si faltan en el catálogo de obras o si hay algún
# problema de formato en la clave que impide el cruce.
# ============================================

# Filtra el DataFrame para mostrar solo las filas con nom_obra nulo
obras_sin_nombre = df[df['nom_obra'].isnull()]

# Muestra la clave de obra, programa y estado de esos registros
obras_sin_nombre[['clave_obra', 'PROGRAMA_PRESUPUESTARIO', 'Estado', 'ORIGINAL', 'MODIFICADO', 'EJERCIDO']]

,clave_obra,PROGRAMA_PRESUPUESTARIO,Estado,ORIGINAL,MODIFICADO,EJERCIDO
39,0,U004,Colima,0,3.640000e+07,3.640000e+07
49,0,U004,Chiapas,0,1.218004e+08,1.218004e+08
73,0,U004,Oficinas Centrales,3000000000,0.000000e+00,0.000000e+00
90,0,U004,Durango,0,3.711400e+08,3.711400e+08
131,0,U004,Guerrero,0,9.324050e+08,9.323769e+08
148,0,U004,Jalisco,0,1.558000e+08,1.558000e+08
185,0,U004,Nayarit,0,2.176300e+08,2.176300e+08
200,0,U004,Oaxaca,0,4.121450e+08,4.121450e+08
217,0,U004,Puebla,0,3.140000e+07,3.140000e+07
258,0,U004,Sonora,0,1.684840e+08,1.684840e+08


In [ ]:
# ============================================
# PASO 4b: Tratar los registros sin nombre de obra (clave 0)
# La clave de obra "0" corresponde a registros del programa
# U004 (Caminos Artesanales), un programa de apoyo/subsidio que
# no se gestiona por proyecto individual de cartera, a diferencia
# de los programas tipo K. Por eso no tiene nombre de obra en el
# catálogo, y se completa con el nombre del programa.
# ============================================

df['nom_obra'] = df['nom_obra'].fillna('Caminos Artesanales')

# Verificación: confirmar que ya no quedan nulos en nom_obra
print("Valores nulos restantes en nom_obra:", df['nom_obra'].isnull().sum())

Valores nulos restantes en nom_obra: 0


In [ ]:
# ============================================
# PASO 5a: Cálculo de variación Modificado vs. Ejercido
# Se calcula la diferencia absoluta y porcentual entre el
# presupuesto modificado y el ejercido, a nivel de cada obra.
# Este cálculo documenta el hallazgo de que, en el cierre anual
# de Cuenta Pública, una parte de las obras muestra reconciliación
# completa entre modificado y ejercido, mientras que otra parte
# presenta variaciones reales y significativas.
#
# Las columnas monetarias se generan también en millones de
# pesos (MDP) para facilitar la lectura, conservando las
# columnas originales en pesos para cálculos posteriores.
# ============================================

# Diferencia absoluta: modificado menos ejercido (en pesos)
df['variacion_mod_ejer'] = df['MODIFICADO'] - df['EJERCIDO']

# Diferencia porcentual: qué % representa esa diferencia sobre el modificado
# Se usa .replace(0, pd.NA) para evitar división entre cero en casos donde MODIFICADO sea 0
df['variacion_mod_ejer_pct'] = (
    df['variacion_mod_ejer'] / df['MODIFICADO'].replace(0, pd.NA)
) * 100

# Columnas en millones de pesos (MDP), redondeadas a 2 decimales
df['MODIFICADO_MDP'] = (df['MODIFICADO'] / 1_000_000).round(2)
df['EJERCIDO_MDP'] = (df['EJERCIDO'] / 1_000_000).round(2)
df['variacion_mod_ejer_MDP'] = (df['variacion_mod_ejer'] / 1_000_000).round(2)

# Estadísticas descriptivas de la variación en millones de pesos
print("Resumen de variación Modificado vs. Ejercido (MDP):")
print(df['variacion_mod_ejer_MDP'].describe())

print("\nResumen de variación porcentual (%):")
print(df['variacion_mod_ejer_pct'].describe())

Resumen de variación Modificado vs. Ejercido (MDP):
count    301.000000
mean       3.083355
std       24.232640
min        0.000000
25%        0.000000
50%        0.000000
75%        0.010000
max      299.170000
Name: variacion_mod_ejer_MDP, dtype: float64

Resumen de variación porcentual (%):
count     300.0
unique    139.0
top         0.0
freq      155.0
Name: variacion_mod_ejer_pct, dtype: float64


In [ ]:
# ============================================
# PASO 5b (simplificado): Top 15 obras con mayor variación
# Modificado vs. Ejercido, en millones de pesos (MDP)
# ============================================

top_variacion = df.sort_values('variacion_mod_ejer_MDP', ascending=False).head(15)

top_variacion[['clave_obra', 'nom_obra', 'PROGRAMA_PRESUPUESTARIO', 'Estado',
               'MODIFICADO_MDP', 'EJERCIDO_MDP', 'variacion_mod_ejer_MDP', 'variacion_mod_ejer_pct']]

,clave_obra,nom_obra,PROGRAMA_PRESUPUESTARIO,Estado,MODIFICADO_MDP,EJERCIDO_MDP,variacion_mod_ejer_MDP,variacion_mod_ejer_pct
288,24092100003,Programa de Conservación de Infraestructura de...,K037,Veracruz,310.20,11.03,299.17,96.444447
139,24092100003,Programa de Conservación de Infraestructura de...,K037,Hidalgo,252.26,8.25,244.01,96.728041
134,24092100003,Programa de Conservación de Infraestructura de...,K037,Guerrero,1666.91,1527.74,139.17,8.348747
68,24096280001,Camino Lim. Edos. Chi. / Son. - Pancho Villa -...,K031,Chihuahua,620.52,552.87,67.64,10.900955
201,24092100003,Programa de Conservación de Infraestructura de...,K037,Oaxaca,1249.28,1188.71,60.57,4.848234
45,25096260001,Construcción del Paso Superior Vehicular Arco Sur,K031,Colima,155.73,124.69,31.04,19.932553
221,24092100003,Programa de Conservación de Infraestructura de...,K037,San Luis Potosí,23.93,0.00,23.93,100.0
44,25096260003,Construcción de Puente Vehicular El Chical,K031,Colima,42.17,20.98,21.19,50.247392
43,25096260001,Construcción del Paso Superior Vehicular Arco Sur,K031,Colima,117.89,100.36,17.53,14.868582
219,24092100003,Programa de Conservación de Infraestructura de...,K037,Querétaro,6.98,0.00,6.98,100.0


In [ ]:
# ============================================
# PASO 5c: Conteo de obras con variación cero vs. distinta de cero
# Cuantifica qué proporción de obras tuvo reconciliación completa
# (variación = 0) frente a las que presentaron alguna diferencia,
# y muestra el total de variación en millones de pesos para cada
# grupo como referencia de magnitud.
# ============================================

sin_variacion = (df['variacion_mod_ejer'] == 0).sum()
con_variacion = (df['variacion_mod_ejer'] != 0).sum()

# Suma total de la variación real (en MDP), solo del grupo con variación distinta de cero
total_variacion_MDP = df.loc[df['variacion_mod_ejer'] != 0, 'variacion_mod_ejer_MDP'].sum().round(2)

print(f"Obras sin variación (modificado = ejercido): {sin_variacion} ({sin_variacion/len(df)*100:.1f}%)")
print(f"Obras con variación real: {con_variacion} ({con_variacion/len(df)*100:.1f}%)")
print(f"Suma total de variación en obras con diferencia real: {total_variacion_MDP} MDP")

Obras sin variación (modificado = ejercido): 156 (51.8%)
Obras con variación real: 145 (48.2%)
Suma total de variación en obras con diferencia real: 928.09 MDP


Interpretación del resultado

51.8% de las obras (156) tuvieron reconciliación completa entre modificado y ejercido.
48.2% de las obras (145) presentaron variación real, sumando 928.09 millones de pesos en conjunto.
La diferencia entre la media (3.08 MDP) y la mediana (0 MDP) confirma que la distribución está sesgada por unos pocos casos extremos (como el de 299.17 MDP que vimos en el top 15) — la mayoría de las obras con variación tienen montos pequeños, pero un grupo reducido concentra montos muy altos.

In [ ]:
# ============================================
# PASO 6: Exportar el DataFrame final para Tableau
# Se guarda el DataFrame ya limpio y enriquecido —con las
# columnas de variación Modificado vs. Ejercido calculadas—
# en un archivo CSV, listo para conectarse a Tableau Public.
# ============================================

df.to_csv('/content/drive/MyDrive/analisis_python/presupuesto_2025_analisis_final.csv', index=False)

print("Archivo exportado correctamente.")

Archivo exportado correctamente.
